# Hybrid nanomaterial selection demonstration
Local demonstration using cached research and a saved TiO2 model. All 15 material families appear. A decision score is not removal percentage, probability or ML accuracy. Unsupported candidates receive no invented score.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.recommend import RecommendationEngine, load_demo
engine = RecommendationEngine()
columns = ['material_id', 'material_name', 'profile_id', 'status', 'preference_tier', 'score_min', 'score_max', 'prediction', 'prediction_target']


## Set water and equipment conditions
These published SMX conditions are a held out model example. The scenario explicitly accepts study profiles and conditional candidates as research proposals. Edit this dictionary or load your own scenario JSON.

In [2]:
scenario = load_demo('demo_ml')
scenario


{'pollutant_id': 'Sulfamethoxazole',
 'light_type': 'UV',
 'water_matrix': 'laboratory_solution',
 'requested_endpoint': 'qualitative_remediation',
 'light_intensity_mW_cm2': 1.51,
 'temperature_C': 25.0,
 'dose_g_L': 0.1,
 'concentration_mg_L': 24.82,
 'pH': 3.0,
 'demo_note': 'First eligible sulfamethoxazole holdout row, otherwise first eligible holdout row; chosen by inputs, not smallest prediction error.',
 'source_row': 256,
 'prediction_endpoint': 'mlogk',
 'include_ml': True,
 'profile_choices': {'M02': 'dataset_unspecified_tio2'},
 'consider_study_profiles': True,
 'allow_conditional_ranking': True,
 'available_equipment': ['mixer', 'centrifuge', 'filtration']}

In [3]:
ranking = engine.recommend(scenario)
display(ranking[columns])
print('Top candidate IDs:', ranking.attrs['summary']['top_candidates'])
print('Result:', ranking.attrs['summary']['recommendation_type'])
print('Score meaning:', ranking.attrs['summary']['score_meaning'])


,material_id,material_name,profile_id,status,preference_tier,score_min,score_max,prediction,prediction_target
0,M02,Titanium dioxide,dataset_unspecified_tio2,conditional,1,55.0,70.0,1.277677,mlogk
1,M03,Zinc oxide,P03,conditional,1,55.0,70.0,NaN,None
2,M08,Carbon nanotubes,P04,conditional,1,55.0,70.0,NaN,None
3,M01,Iron oxide magnetite,None,insufficient_evidence,<NA>,NaN,NaN,NaN,None
4,M04,Silver nanoparticles,None,insufficient_evidence,<NA>,NaN,NaN,NaN,None
5,M05,Nanoscale zero valent iron,None,insufficient_evidence,<NA>,NaN,NaN,NaN,None
6,M06,Graphene oxide,None,insufficient_evidence,<NA>,NaN,NaN,NaN,None
7,M07,Reduced graphene oxide,None,insufficient_evidence,<NA>,NaN,NaN,NaN,None
8,M09,Nano alumina,None,insufficient_evidence,<NA>,NaN,NaN,NaN,None
9,M10,Magnesium oxide,None,insufficient_evidence,<NA>,NaN,NaN,NaN,None


Top candidate IDs: ['M02', 'M03', 'M08']
Result: uncertain shortlist
Score meaning: Project chosen evidence applicability and practical setup preference score. Not a removal percentage, efficiency prediction or probability.


## Inspect the prediction and its limits
The numerical mlogk output belongs to the unspecified TiO2 dataset profile. It is not assigned to the source's brookite or anatase formulations, and it does not break the tied evidence shortlist.

In [4]:
tio2 = ranking.set_index('material_id').loc['M02']
print(tio2['reason'])
print('Prediction profile:', tio2['prediction_profile_id'])
print('Sources:', tio2['source_urls'])
print('Alternate weights:', ranking.attrs['summary']['alternative_weight_results'])


Eligible experimental ML estimate attached separately; no numerical treatment efficiency or extra ranking score is inferred. Catalyst grade and recovery are unmodelled, so this remains a conditional research candidate. Preliminary input group holdout; missing study identifiers prevent independent study validation. Only TiO2 family and known pollutants; catalyst phase, grade, size, surface area and reactor geometry are unmodelled. Only the reported mlogk target is predicted; logarithm base is unverified so no physical k or removal percentage is returned. Laboratory solution is a demonstration assumption; water matrix was not recorded per row, and real water effects are unvalidated. Range and nearest condition checks are heuristics, not calibrated prediction intervals or proof of applicability. Pollutant identity is encoded categorically; no generalisation to unseen chemicals is supported.
Prediction profile: dataset_unspecified_tio2
Sources: https://github.com/mahyavs/GNN_photodegradati

## Change available light
The documented photocatalytic routes cannot run in the dark. CNT adsorption remains a conditional research candidate; a unique policy candidate is not a performance guarantee.

In [5]:
dark = engine.recommend(dict(scenario, light_type='dark'))
display(dark[columns])
print(dark.attrs['summary']['recommendation_type'], dark.attrs['summary']['top_candidates'])


,material_id,material_name,profile_id,status,preference_tier,score_min,score_max,prediction,prediction_target
0,M08,Carbon nanotubes,P04,conditional,1,55.0,70.0,None,None
1,M02,Titanium dioxide,dataset_unspecified_tio2,incompatible_with_setup,<NA>,NaN,NaN,None,None
2,M03,Zinc oxide,P03,incompatible_with_setup,<NA>,NaN,NaN,None,None
3,M01,Iron oxide magnetite,None,insufficient_evidence,<NA>,NaN,NaN,None,None
4,M04,Silver nanoparticles,None,insufficient_evidence,<NA>,NaN,NaN,None,None
5,M05,Nanoscale zero valent iron,None,insufficient_evidence,<NA>,NaN,NaN,None,None
6,M06,Graphene oxide,None,insufficient_evidence,<NA>,NaN,NaN,None,None
7,M07,Reduced graphene oxide,None,insufficient_evidence,<NA>,NaN,NaN,None,None
8,M09,Nano alumina,None,insufficient_evidence,<NA>,NaN,NaN,None,None
9,M10,Magnesium oxide,None,insufficient_evidence,<NA>,NaN,NaN,None,None


unique conditional policy candidate ['M08']


## Source matched lead example
This case deliberately matches the published hydroxyapatite batch observation. The 100 point policy score means complete checked rubric support, not 100 percent removal. The observed removal and source limitations remain separate.

In [6]:
lead = engine.recommend(load_demo('demo_lead'))
display(lead[columns])
print('Policy winner:', lead.attrs['summary']['unique_policy_winner'])
print(lead.set_index('material_id').loc['M13', 'reason'])
print('Source:', lead.set_index('material_id').loc['M13', 'source_urls'])


,material_id,material_name,profile_id,status,preference_tier,score_min,score_max,prediction,prediction_target
0,M13,Nano hydroxyapatite,P09,supported,1,100.0,100.0,None,None
1,M01,Iron oxide magnetite,P07,conditional,2,62.5,62.5,None,None
2,M06,Graphene oxide,P08,conditional,2,55.0,70.0,None,None
3,M15,Chitosan nanoparticles,P10,conditional,2,55.0,70.0,None,None
4,M10,Magnesium oxide,P11,conditional,3,42.5,57.5,None,None
5,M02,Titanium dioxide,None,insufficient_evidence,<NA>,NaN,NaN,None,None
6,M03,Zinc oxide,None,insufficient_evidence,<NA>,NaN,NaN,None,None
7,M04,Silver nanoparticles,None,insufficient_evidence,<NA>,NaN,NaN,None,None
8,M05,Nanoscale zero valent iron,None,insufficient_evidence,<NA>,NaN,NaN,None,None
9,M07,Reduced graphene oxide,None,insufficient_evidence,<NA>,NaN,NaN,None,None


Policy winner: M13
A source observation matches the five checked operating inputs and reports the requested removal target; this is literature support, not a new prediction. Closest extracted observation E014: matching fields pH, concentration_mg_L, dose_g_L, temperature_C, time_min; different fields none; unknown fields none. The reported adsorption route does not require illumination. Centrifugation in batch measurement; packed bed retention in column configuration
Source: https://pmc.ncbi.nlm.nih.gov/articles/PMC9067131/


## Unsupported pollutant
No material performance or rank is fabricated when the pollutant is outside the evidence catalogue.

In [7]:
unknown = engine.recommend(load_demo('demo_unknown'))
display(unknown[['material_id', 'material_name', 'status', 'score_min', 'prediction']])
print(unknown.attrs['summary']['recommendation_type'])


,material_id,material_name,status,score_min,prediction
0,M01,Iron oxide magnetite,insufficient_evidence,None,None
1,M02,Titanium dioxide,insufficient_evidence,None,None
2,M03,Zinc oxide,insufficient_evidence,None,None
3,M04,Silver nanoparticles,insufficient_evidence,None,None
4,M05,Nanoscale zero valent iron,insufficient_evidence,None,None
5,M06,Graphene oxide,insufficient_evidence,None,None
6,M07,Reduced graphene oxide,insufficient_evidence,None,None
7,M08,Carbon nanotubes,insufficient_evidence,None,None
8,M09,Nano alumina,insufficient_evidence,None,None
9,M10,Magnesium oxide,insufficient_evidence,None,None


no justified ranking


## What the prototype establishes
The local system integrates conditional evidence screening, fixed decision rules, interval ranking and one evaluated ML module. It covers all 15 families as candidates, with numerical prediction only in its documented subset. No independent study validation, lab validation or drinking water certification is claimed. Read outputs/hybrid_integration.md and outputs/model_evaluation.md for scope and exact metrics.